## 批量处理 vertical_well_las_target（对象流版本）

本笔记改为统一对象流：

- 先将 LAS 转为每井 Dict[str, grid.Log]
- 调用 replace_constant_value_intervals_in_log_dicts 执行连续常值区间替换
- 调用 export_logsets_to_las 统一导出 LAS

判定规则与旧版一致：

- 严格相等判定
- 连续长度 >= min_run_length 即替换为 anomaly_value
- 缺失值 (-999.0, -999.25, -99999, NaN) 不参与判定


In [ ]:
import sys
from pathlib import Path

import lasio
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "src").exists() and (p / "data").exists():
            return p
    raise RuntimeError("未找到仓库根目录（需要同时包含 src 和 data 目录）。")


cwd = Path.cwd().resolve()
ROOT = find_repo_root(cwd)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from cup.petrel.export import export_logsets_to_las
from cup.petrel.load import extract_any_log_from_las
from cup.well.process import replace_constant_value_intervals_in_log_dicts

In [ ]:
input_dir = ROOT / "data" / "vertical_well_las_target"
output_dir = ROOT / "data" / "vertical_well_las_target_clear"

min_run_length = 8
anomaly_value = -999.25

In [ ]:
# Windows 大小写不敏感，按解析后的绝对路径去重，避免同一文件被重复处理。
las_candidates = [*input_dir.glob("*.las"), *input_dir.glob("*.Las"), *input_dir.glob("*.LAS")]
las_files = sorted({p.resolve(): p for p in las_candidates}.values(), key=lambda p: p.name.lower())

if not las_files:
    raise RuntimeError(f"目录下未找到 LAS 文件: {input_dir}")

raw_logsets = {}
import_skipped_wells = []
import_skipped_curves = []

for las_path in las_files:
    well_name = las_path.stem
    try:
        las = lasio.read(str(las_path))
        curve_names_in_file = [str(c) for c in las.df().columns]

        logs = {}
        for curve_name in curve_names_in_file:
            try:
                logs[curve_name] = extract_any_log_from_las(las, curve_name)
            except Exception as exc:
                import_skipped_curves.append({"well": well_name, "curve": curve_name, "reason": str(exc)})

        if not logs:
            import_skipped_wells.append({"well": well_name, "reason": "无可导入曲线"})
            continue

        raw_logsets[well_name] = logs
    except Exception as exc:
        import_skipped_wells.append({"well": well_name, "reason": str(exc)})

process_result = replace_constant_value_intervals_in_log_dicts(
    logsets=raw_logsets,
    min_run_length=min_run_length,
    anomaly_value=anomaly_value,
)

processed_logsets = process_result["processed_logsets"]

curve_suffix = "_CLAER"
export_logsets = {}
for well_name, logs in processed_logsets.items():
    renamed_logs = {}
    for curve_name, log in logs.items():
        new_curve_name = curve_name if curve_name.endswith(curve_suffix) else f"{curve_name}{curve_suffix}"
        renamed_logs[new_curve_name] = log
    export_logsets[well_name] = renamed_logs

export_result = export_logsets_to_las(
    logsets=export_logsets,
    output_dir=output_dir,
    null_value=anomaly_value,
)

summary_df = pd.DataFrame(
    {
        "input_file_count": [len(las_files)],
        "imported_well_count": [len(raw_logsets)],
        "import_skipped_well_count": [len(import_skipped_wells)],
        "import_skipped_curve_count": [len(import_skipped_curves)],
        "target_curve_count": [process_result["target_curve_count"]],
        "curves_with_replacement": [process_result["curves_with_replacement"]],
        "total_replaced_points": [process_result["total_replaced_points"]],
        "exported_file_count": [len(export_result["exported_files"])],
        "export_skipped_well_count": [len(export_result["skipped_wells"])],
        "export_skipped_curve_count": [len(export_result["skipped_curves"])],
        "output_dir": [str(output_dir)],
    }
)

summary_df

In [ ]:
replacement_detail_df = pd.DataFrame(process_result["curve_reports"])

if replacement_detail_df.empty:
    print("所有井处理完成，但未发现达到阈值的连续相同值区间。")
else:
    replacement_detail_df = replacement_detail_df.sort_values(
        ["well", "replaced_points"], ascending=[True, False]
    ).reset_index(drop=True)
    replacement_detail_df

print(f"导入阶段跳过井数量: {len(import_skipped_wells)}")
print(f"导入阶段跳过曲线数量: {len(import_skipped_curves)}")
print(f"导出阶段跳过井数量: {len(export_result['skipped_wells'])}")
print(f"导出阶段跳过曲线数量: {len(export_result['skipped_curves'])}")

(
    import_skipped_wells[:5],
    import_skipped_curves[:5],
    export_result["skipped_wells"][:5],
    export_result["skipped_curves"][:5],
)